In [ ]:
"""
00_build_cache
==============
One-time notebook to populate the shared wearables cache bucket.

Run modalities in this order (smallest → largest).

    sleep_summary       ~minutes
    sleep_onset_offset  ~minutes
    steps_hourly        ~30-60 min
    hr_hourly           ~3-6 hours

Each modality is fully independent and crash-safe.
Re-running a completed modality is a no-op (all stages are checkpointed).
"""

# ── 0. Environment ─────────────────────────────────────────────────────────────
import sys, os
sys.path.insert(0, os.path.abspath(os.path.join(os.getcwd(), '..')))
from src.utils import *

from src.wearables_cache import run_pipeline, cache_tree

cache_tree()

In [ ]:
# ── 1. Choose modality ─────────────────────────────────────────────────────────
# Run one modality at a time and inspect the output before the next, in the order
# sleep_summary → sleep_onset_offset → steps_hourly → hr_hourly.
MODALITY = "sleep_onset_offset"      # change per run

In [ ]:
# ── 2. Run ────────────────────────────────────────────────────────────────────
run_pipeline(
    modality       = MODALITY,
    client         = client,
    dataset        = DATASET,
    overwrite_meta = False,        # True only if you need to refresh pid list
)

In [ ]:
# ── 3. Example fetch (smoke test after each modality) ─────────────────────────
from src.wearables_cache import fetch_wearables

# Draw the smoke-test sample from the cache index at run time, so no participant
# identifiers are stored in this notebook.
cache_index = load_cache_parquet('meta/index.parquet')
SAMPLE_PIDS = cache_index['person_id'].sample(n=50, random_state=0).tolist()

df = fetch_wearables(
    pids      = SAMPLE_PIDS,
    modality  = MODALITY,
    date_range = None,             # no filter → full recording
)
print(df.dtypes)
print(f"{len(df):,} rows for {df['person_id'].nunique():,} participants")